# 06 — Mixed-Integer Linear Programming (MILP) Energy Scheduling
Formulates the 24-hour ahead microgrid dispatch schedule with Pyomo to minimize fuel consumption, battery degradation, and generator cycling costs.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.optimization.milp_model import solve_milp


In [ ]:
with open(PROJECT_ROOT / "configs" / "battery_config.yaml") as f:
    b_cfg = yaml.safe_load(f)["battery"]
with open(PROJECT_ROOT / "configs" / "diesel_config.yaml") as f:
    d_cfg = yaml.safe_load(f)["diesel"]
params = {"battery": b_cfg, "diesel": d_cfg}

# Load a 24-hour forecast window
master_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "master_dataset.csv")
sample_24h = master_df.iloc[1000:1024].copy()

load = sample_24h["load_kw"].values
critical = sample_24h["critical_load_kw"].values
solar = sample_24h["solar_generation_kw"].values
wind = sample_24h["wind_generation_kw"].values

schedule_df, solved = solve_milp(load, critical, solar, wind, params)
print("MILP Solved Optimally?", solved)
schedule_df.head()


In [ ]:
# Visualizing the 24-Hour Dispatch Schedule
plt.figure(figsize=(12, 6))
plt.plot(schedule_df['solar_use'], label='Solar Used (kW)', color='orange')
plt.plot(schedule_df['wind_use'], label='Wind Used (kW)', color='dodgerblue')
plt.plot(schedule_df['diesel_gen'], label='Diesel Gen (kW)', color='crimson')
plt.plot(schedule_df['battery_discharge'], label='Battery Discharge (kW)', color='purple', linestyle='--')
plt.plot(load, label='Total Demand (kW)', color='black', lw=2)
plt.title("24-Hour Optimal Energy Scheduling (MILP)")
plt.xlabel("Hour of Horizon")
plt.ylabel("Power (kW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
